<a href="https://colab.research.google.com/github/davidsmmcomercial-max/befly_hotel_booking_pipeline/blob/main/befly_hotel_booking_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pyspark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BeFly - Hotel Booking Pipeline")
    .master("local[*]")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

spark

In [3]:
!ls -lh

total 8.0K
drwxr-xr-x 6 root root 4.0K May 11 17:56 data
drwxr-xr-x 1 root root 4.0K May  6 13:29 sample_data


In [4]:
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/bronze", exist_ok=True)
os.makedirs("data/silver", exist_ok=True)
os.makedirs("data/gold", exist_ok=True)

In [5]:
import shutil
from pathlib import Path

raw_path = Path("data/raw")

for file_name in ["hotel_bookings.csv", "country_metadata.csv", "hotel_metadata.csv"]:
    if Path(file_name).exists():
        shutil.move(file_name, raw_path / file_name)

!ls -lh data/raw

total 17M
-rw-r--r-- 1 root root 6.1K May 11 17:58 country_metadata.csv
-rw-r--r-- 1 root root  17M May 11 17:58 hotel_bookings.csv
-rw-r--r-- 1 root root  117 May 11 17:58 hotel_metadata.csv


In [6]:
from pyspark.sql import functions as F

bookings_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("nullValue", "NULL")
    .csv("data/raw/hotel_bookings.csv")
)

countries_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("data/raw/country_metadata.csv")
)

hotels_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("data/raw/hotel_metadata.csv")
)

In [7]:
print(f"Bookings: {bookings_df.count()} registros")
print(f"Countries: {countries_df.count()} registros")
print(f"Hotels: {hotels_df.count()} registros")

Bookings: 119390 registros
Countries: 252 registros
Hotels: 2 registros


In [8]:
(
    bookings_df.write
    .mode("overwrite")
    .partitionBy("arrival_date_year")
    .parquet("data/bronze/bookings")
)

(
    countries_df.write
    .mode("overwrite")
    .parquet("data/bronze/countries")
)

(
    hotels_df.write
    .mode("overwrite")
    .parquet("data/bronze/hotels")
)

In [9]:
!find data/bronze -type d

data/bronze
data/bronze/bookings
data/bronze/bookings/arrival_date_year=2017
data/bronze/bookings/arrival_date_year=2016
data/bronze/bookings/arrival_date_year=2015
data/bronze/countries
data/bronze/hotels


In [10]:
from pyspark.sql import functions as F

bronze_bookings = spark.read.parquet("data/bronze/bookings")
bronze_countries = spark.read.parquet("data/bronze/countries")
bronze_hotels = spark.read.parquet("data/bronze/hotels")

print(bronze_bookings.count())

119390


In [11]:
from pathlib import Path
from pyspark.sql import functions as F

BASE_DIR = Path("/content")
BRONZE_DIR = BASE_DIR / "data" / "bronze"
SILVER_DIR = BASE_DIR / "data" / "silver"

print("Lendo dados da camada Bronze...")

bronze_bookings = spark.read.parquet(str(BRONZE_DIR / "bookings"))
bronze_countries = spark.read.parquet(str(BRONZE_DIR / "countries"))
bronze_hotels = spark.read.parquet(str(BRONZE_DIR / "hotels"))

month_map = F.create_map(
    [F.lit(x) for x in [
        "January", 1,
        "February", 2,
        "March", 3,
        "April", 4,
        "May", 5,
        "June", 6,
        "July", 7,
        "August", 8,
        "September", 9,
        "October", 10,
        "November", 11,
        "December", 12,
    ]]
)

int_cols = [
    "is_canceled",
    "lead_time",
    "arrival_date_year",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "required_car_parking_spaces",
    "total_of_special_requests",
]

silver_df = bronze_bookings

print("Convertendo tipos...")

for col_name in int_cols:
    silver_df = silver_df.withColumn(
        col_name,
        F.expr(f"try_cast({col_name} as int)")
    )

silver_df = (
    silver_df
    .withColumn("adr", F.expr("try_cast(adr as double)"))
    .withColumn("reservation_status_date", F.to_date("reservation_status_date"))
    .withColumn("children", F.coalesce(F.col("children"), F.lit(0)))
    .withColumn("country", F.coalesce(F.col("country"), F.lit("UNK")))
    .withColumn("arrival_date_month_num", month_map[F.col("arrival_date_month")])
    .withColumn(
        "arrival_date",
        F.to_date(
            F.concat_ws(
                "-",
                F.col("arrival_date_year"),
                F.lpad(F.col("arrival_date_month_num"), 2, "0"),
                F.lpad(F.col("arrival_date_day_of_month"), 2, "0"),
            )
        )
    )
    .withColumn(
        "total_nights",
        F.col("stays_in_weekend_nights") + F.col("stays_in_week_nights")
    )
    .withColumn(
        "total_guests",
        F.col("adults") + F.col("children") + F.col("babies")
    )
    .withColumn(
        "is_family",
        F.when((F.col("children") > 0) | (F.col("babies") > 0), 1).otherwise(0)
    )
    .withColumn(
        "is_long_stay",
        F.when(F.col("total_nights") > 7, 1).otherwise(0)
    )
    .withColumn(
        "revenue",
        F.when(
            F.col("is_canceled") == 0,
            F.col("adr") * F.col("total_nights")
        ).otherwise(0)
    )
    .withColumn(
        "booking_status",
        F.when(F.col("reservation_status") == "Canceled", "Canceled")
        .when(F.col("reservation_status") == "No-Show", "NoShow")
        .when(F.col("reservation_status") == "Check-Out", "CheckedOut")
        .otherwise("Unknown")
    )
)

print("Aplicando filtros de qualidade...")

before_count = silver_df.count()

silver_df = silver_df.filter(F.col("total_guests") > 0)
after_guest_filter_count = silver_df.count()

silver_df = silver_df.filter(F.col("adr") >= 0)
after_adr_filter_count = silver_df.count()

print(f"Registros antes dos filtros: {before_count}")
print(f"Removidos sem hóspedes: {before_count - after_guest_filter_count}")
print(f"Removidos com ADR negativo: {after_guest_filter_count - after_adr_filter_count}")
print(f"Registros finais antes do enriquecimento: {after_adr_filter_count}")

print("Enriquecendo com tabelas de referência...")

silver_enriched_df = (
    silver_df
    .join(
        bronze_countries,
        silver_df["country"] == bronze_countries["country_code"],
        "left"
    )
    .join(
        bronze_hotels,
        silver_df["hotel"] == bronze_hotels["hotel_name"],
        "left"
    )
    .withColumnRenamed("city", "hotel_city")
    .withColumnRenamed("star_rating", "hotel_star_rating")
    .withColumnRenamed("opened_year", "hotel_opened_year")
    .drop("country_code", "hotel_name")
)

silver_before_join_count = silver_df.count()
silver_after_join_count = silver_enriched_df.count()

print(f"Registros antes do enriquecimento: {silver_before_join_count}")
print(f"Registros após enriquecimento: {silver_after_join_count}")

if silver_before_join_count == silver_after_join_count:
    print("Validação OK: joins não duplicaram registros.")
else:
    print("Atenção: joins alteraram a quantidade de registros.")

print("Gravando camada Silver...")

(
    silver_enriched_df.write
    .mode("overwrite")
    .partitionBy("arrival_date_year", "arrival_date_month_num")
    .parquet(str(SILVER_DIR / "bookings_enriched"))
)

print("Camada Silver criada com sucesso.")
print(f"Path: {SILVER_DIR / 'bookings_enriched'}")

Lendo dados da camada Bronze...
Convertendo tipos...
Aplicando filtros de qualidade...
Registros antes dos filtros: 119390
Removidos sem hóspedes: 180
Removidos com ADR negativo: 1
Registros finais antes do enriquecimento: 119209
Enriquecendo com tabelas de referência...
Registros antes do enriquecimento: 119209
Registros após enriquecimento: 119209
Validação OK: joins não duplicaram registros.
Gravando camada Silver...
Camada Silver criada com sucesso.
Path: /content/data/silver/bookings_enriched


In [12]:
silver_enriched_df = (
    silver_df
    .join(
        bronze_countries,
        silver_df["country"] == bronze_countries["country_code"],
        "left"
    )
    .join(
        bronze_hotels,
        silver_df["hotel"] == bronze_hotels["hotel_name"],
        "left"
    )
    .withColumnRenamed("city", "hotel_city")
    .withColumnRenamed("star_rating", "hotel_star_rating")
    .withColumnRenamed("opened_year", "hotel_opened_year")
    .drop("country_code", "hotel_name")
)

print(f"Registros Silver enriquecida: {silver_enriched_df.count()}")

Registros Silver enriquecida: 119209


In [13]:
(
    silver_enriched_df.write
    .mode("overwrite")
    .partitionBy("arrival_date_year", "arrival_date_month_num")
    .parquet("data/silver/bookings_enriched")
)

In [14]:
!find data/silver -type d | head -30

data/silver
data/silver/bookings_enriched
data/silver/bookings_enriched/arrival_date_year=2017
data/silver/bookings_enriched/arrival_date_year=2017/arrival_date_month_num=5
data/silver/bookings_enriched/arrival_date_year=2017/arrival_date_month_num=3
data/silver/bookings_enriched/arrival_date_year=2017/arrival_date_month_num=8
data/silver/bookings_enriched/arrival_date_year=2017/arrival_date_month_num=6
data/silver/bookings_enriched/arrival_date_year=2017/arrival_date_month_num=1
data/silver/bookings_enriched/arrival_date_year=2017/arrival_date_month_num=2
data/silver/bookings_enriched/arrival_date_year=2017/arrival_date_month_num=7
data/silver/bookings_enriched/arrival_date_year=2017/arrival_date_month_num=4
data/silver/bookings_enriched/arrival_date_year=2016
data/silver/bookings_enriched/arrival_date_year=2016/arrival_date_month_num=5
data/silver/bookings_enriched/arrival_date_year=2016/arrival_date_month_num=3
data/silver/bookings_enriched/arrival_date_year=2016/arrival_date_month_

In [15]:
silver_gold_df = spark.read.parquet("data/silver/bookings_enriched")

In [16]:
revenue_by_hotel_month = (
    silver_gold_df
    .groupBy(
        "hotel",
        "arrival_date_year",
        "arrival_date_month_num"
    )
    .agg(
        F.count("*").alias("total_bookings"),

        F.sum(
            F.when(F.col("is_canceled") == 0, 1).otherwise(0)
        ).alias("effective_bookings"),

        F.sum(
            F.when(F.col("is_canceled") == 1, 1).otherwise(0)
        ).alias("cancelled_bookings"),

        F.round(
            F.sum("revenue"),
            2
        ).alias("total_revenue_eur"),

        F.round(
            F.avg(
                F.when(
                    F.col("is_canceled") == 0,
                    F.col("adr")
                )
            ),
            2
        ).alias("avg_adr_eur"),

        F.sum(
            F.when(
                F.col("is_canceled") == 0,
                F.col("total_nights")
            ).otherwise(0)
        ).alias("total_nights_sold"),

        F.round(
            (
                F.sum(
                    F.when(F.col("is_canceled") == 1, 1).otherwise(0)
                ) / F.count("*")
            ) * 100,
            2
        ).alias("cancellation_rate_pct")
    )
)

In [17]:
(
    revenue_by_hotel_month.write
    .mode("overwrite")
    .parquet("data/gold/revenue_by_hotel_month")
)

In [18]:
revenue_by_hotel_month.show(truncate=False)

+------------+-----------------+----------------------+--------------+------------------+------------------+-----------------+-----------+-----------------+---------------------+
|hotel       |arrival_date_year|arrival_date_month_num|total_bookings|effective_bookings|cancelled_bookings|total_revenue_eur|avg_adr_eur|total_nights_sold|cancellation_rate_pct|
+------------+-----------------+----------------------+--------------+------------------+------------------+-----------------+-----------+-----------------+---------------------+
|City Hotel  |2016             |2                     |2365          |1436              |929               |318750.98        |82.33      |3857             |39.28                |
|Resort Hotel|2017             |5                     |1757          |1212              |545               |444897.6         |82.78      |5352             |31.02                |
|City Hotel  |2015             |9                     |3524          |1982              |1542            

In [19]:
cancellation_by_segment = (
    silver_gold_df
    .groupBy(
        "market_segment",
        "customer_type",
        "distribution_channel"
    )
    .agg(
        F.count("*").alias("total_bookings"),
        F.sum(
            F.when(F.col("is_canceled") == 1, 1).otherwise(0)
        ).alias("cancelled_bookings"),
        F.round(
            F.sum(
                F.when(F.col("is_canceled") == 1, 1).otherwise(0)
            ) / F.count("*"),
            4
        ).alias("cancellation_rate"),
        F.round(F.avg("lead_time"), 2).alias("avg_lead_time"),
        F.round(
            F.avg("total_of_special_requests"),
            2
        ).alias("avg_total_special_requests")
    )
    .orderBy(F.desc("cancellation_rate"))
)

cancellation_by_segment.show(20, truncate=False)

+--------------+---------------+--------------------+--------------+------------------+-----------------+-------------+--------------------------+
|market_segment|customer_type  |distribution_channel|total_bookings|cancelled_bookings|cancellation_rate|avg_lead_time|avg_total_special_requests|
+--------------+---------------+--------------------+--------------+------------------+-----------------+-------------+--------------------------+
|Groups        |Contract       |Corporate           |4             |4                 |1.0              |114.0        |0.0                       |
|Direct        |Transient-Party|Undefined           |1             |1                 |1.0              |1.0          |1.0                       |
|Undefined     |Transient-Party|Undefined           |2             |2                 |1.0              |1.5          |1.5                       |
|Online TA     |Transient-Party|Undefined           |1             |1                 |1.0              |8.0          

In [20]:
(
    cancellation_by_segment.write
    .mode("overwrite")
    .parquet("data/gold/cancellation_by_segment")
)

In [21]:
(
    bronze_bookings
    .groupBy("distribution_channel")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+--------------------+-----+
|distribution_channel|count|
+--------------------+-----+
|TA/TO               |97870|
|Direct              |14645|
|Corporate           |6677 |
|GDS                 |193  |
|Undefined           |5    |
+--------------------+-----+



In [22]:
bronze_undefined = (
    bronze_bookings
    .filter(F.col("distribution_channel") == "Undefined")
    .count()
)

print(bronze_undefined)

5


In [23]:
silver_undefined = (
    silver_gold_df
    .filter(F.col("distribution_channel") == "Undefined")
    .count()
)

print(silver_undefined)

5


In [24]:
top_countries_by_revenue = (
    silver_gold_df
    .filter(F.col("is_canceled") == 0)
    .groupBy(
        "country",
        "country_name",
        "continent"
    )
    .agg(
        F.count("*").alias("effective_bookings"),
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.round(F.avg("revenue"), 2).alias("avg_ticket"),
        F.round(F.avg("lead_time"), 2).alias("avg_lead_time")
    )
    .orderBy(F.desc("total_revenue"))
    .limit(20)
)

top_countries_by_revenue.show(truncate=False)

+-------+------------------+-------------+------------------+-------------+----------+-------------+
|country|country_name      |continent    |effective_bookings|total_revenue|avg_ticket|avg_lead_time|
+-------+------------------+-------------+------------------+-------------+----------+-------------+
|PRT    |Portugal          |Europe       |20977             |5540431.72   |264.12    |49.46        |
|GBR    |United Kingdom    |Europe       |9667              |4110896.61   |425.25    |126.46       |
|FRA    |France            |Europe       |8468              |3098418.27   |365.9     |77.63        |
|ESP    |Spain             |Europe       |6383              |2246305.29   |351.92    |44.36        |
|DEU    |Germany           |Europe       |6067              |2068219.65   |340.9     |139.34       |
|IRL    |Ireland           |Europe       |2542              |1240002.35   |487.81    |112.28       |
|ITA    |Italy             |Europe       |2428              |866439.42    |356.85    |80.35

In [25]:
(
    top_countries_by_revenue.write
    .mode("overwrite")
    .parquet("data/gold/top_countries_by_revenue")
)

In [26]:
guest_stay_profile = (
    silver_gold_df
    .groupBy(
        "hotel",
        "customer_type"
    )
    .agg(
        F.count("*").alias("total_bookings"),
        F.round(F.avg("total_nights"), 2).alias("avg_total_nights"),
        F.round(F.avg("total_guests"), 2).alias("avg_total_guests"),
        F.round(F.avg("is_long_stay") * 100, 2).alias("pct_long_stay"),
        F.round(F.avg("is_family") * 100, 2).alias("pct_family")
    )
    .orderBy("hotel", "customer_type")
)

guest_stay_profile.show(truncate=False)

+------------+---------------+--------------+----------------+----------------+-------------+----------+
|hotel       |customer_type  |total_bookings|avg_total_nights|avg_total_guests|pct_long_stay|pct_family|
+------------+---------------+--------------+----------------+----------------+-------------+----------+
|City Hotel  |Contract       |2296          |2.82            |2.0             |1.35         |5.62      |
|City Hotel  |Group          |291           |2.51            |1.84            |0.69         |5.15      |
|City Hotel  |Transient      |59272         |3.07            |2.01            |1.86         |8.3       |
|City Hotel  |Transient-Party|17304         |2.66            |1.74            |0.51         |1.97      |
|Resort Hotel|Contract       |1776          |8.56            |1.99            |39.81        |3.6       |
|Resort Hotel|Group          |283           |3.22            |3.05            |6.71         |6.01      |
|Resort Hotel|Transient      |30204         |4.18      